### Open ended stub antenna

An open-ended stub antenna is a simple radiating element consisting of a transmission line (typically microstrip or coaxial cable) with one end open-circuited, allowing electromagnetic energy to radiate directly into free space.

In [1]:
import gmsh
import math

from palacetoolkit.viz import view_mesh
from palacetoolkit.mesh import (
    Entity, 
    run_entity_pipeline, 
    generate_3d_mesh, 
    create_graded_mesh
)
from palacetoolkit.simulation import Simulation

### Paramters
- l1 : Ground plane length along x-axis, specified as a scalar in meters
- w1 : Ground plane width along y-axis, specified as a scalar in meters
- h : Patch height along z-axis, specified as a scalar in meters.

- strip_line_length : Notch length along x-axis, specified as a scalar in meters. 
- strip_lined_width: Notch width along y-axis, specified as a scalar in meters.
- air_height : Air-domain height reference for sizing the enclosing sphere, specified as a scalar in meters.
- air_margin : Air-domain margin along x and y axes used for sphere sizing, specified as a scalar in meters.
- freq  : Simulation frequency in GHz, specified as a scalar.
- filename : Output mesh filename, specified as a string.

In [2]:
l1: float = 0.06
w1: float = 0.06
strip_line_length: float = 0.04
strip_line_width: float = 0.002
h: float = 0.0013
air_height: float = 0.025    
air_margin: float = 0.025    
freq: float = 3.3
filename: str = "open_ended_antenna.msh"

wavelength = 3e8 / (freq * 1e9)

### Model initialization

In [3]:
gmsh.initialize()
gmsh.model.add("patch_antenna")
kernel = gmsh.model.occ

### Geometry construction

In [4]:
# Total domain bounds
total_xmin = -l1/2 - air_margin
total_xmax = l1/2 + air_margin
total_ymin = -w1/2 - air_margin
total_ymax = w1/2 + air_margin
total_zmax = h + air_height

# Substrate box
substrate = kernel.addBox(-l1/2, -w1/2, 0, l1, w1, h)

# Ground plane
ground_plane = kernel.addRectangle(-l1/2, -w1/2, 0, l1, w1)

# Top conductor
strip_line_1 = kernel.addRectangle(-l1/2, -strip_line_width/2, h, strip_line_length, strip_line_width)

# gap bewteen the ground plane and the bottom of the lumped port.
gap = 0

# lumped port
lumped_port = kernel.addRectangle(-l1/2 + gap, -strip_line_width/2, 0, h - gap, strip_line_width)
kernel.rotate([(2, lumped_port)], -l1/2, 0, 0, 0, 1, 0, -math.pi/2)

# Air sphere
airsphere_radius = max(abs(total_xmin), abs(total_xmax), abs(total_ymin), abs(total_ymax), total_zmax)
air_sphere = kernel.addSphere(0.0, 0.0, 0.0, airsphere_radius)

# Synchronize everything!
kernel.synchronize()

In [5]:
# Define the entities which later will become the physical groups.
entities = [
    Entity("air_sphere", dim=3, btype="dielectric", mesh_order=2, tags=[air_sphere], eps_r=1.0, mu_r=1.0, loss_tan=0.0),
    Entity("substrate", dim=3, btype="dielectric", mesh_order=1, tags=[substrate], eps_r=2.2, mu_r=1.0, loss_tan=0.0009),
    Entity("top_conductor", dim=2, btype="pec", mesh_order=1, tags=[strip_line_1]),
    Entity("ground_plane", dim=2, btype="pec", mesh_order=1, tags=[ground_plane]),
    Entity("lumped_port", dim=2, btype="lumped_port", mesh_order=0, tags=[lumped_port], R=50.0, direction="+Z")
]

# Boolean operations to guarantee a nice mesh, algo it returns the
# physical group map.
pg_map = run_entity_pipeline(entities)

# Refine near the top conductor and locally the lumped port
create_graded_mesh(  wavelength, 
                     ppw_near=50, 
                     ppw_far=30, 
                     set_as_background=True,
                     local_refinements = {entities[-1].dimtags[0]: 150})
# Mesh sizes
mesh_sizes = {
    "substrate": wavelength / 12,
    "air_sphere": wavelength / 4,
    "lumped_port": wavelength / 150,
    "ground_plane" : wavelength / 10,
    "top_conductor": wavelength / 50
}

# Generate the 3d mesh.
generate_3d_mesh(entities, mesh_sizes, filename, optimize = True)

  Physical group 'air_sphere' (dim=3): pg=1, tags=[2]
  Physical group 'substrate' (dim=3): pg=2, tags=[1]
  Physical group 'top_conductor' (dim=2): pg=3, tags=[8]
  Physical group 'ground_plane' (dim=2): pg=4, tags=[7]
  Physical group 'lumped_port' (dim=2): pg=5, tags=[9]
  Physical group 'air_sphere__None' (dim=2): pg=6, tags=[16]
  Physical group 'air_sphere__substrate' (dim=2): pg=7, tags=[10, 11, 13, 15, 14, 12]
  ignoring 3 curves from {'air_sphere__None'}
  global: 21 curves, SizeMin=0.0018
  ppw_near=50  ppw_far=30
  SizeMax=0.0030  transition=0.0227
  local: dimtag=(2, 9), ppw=150, SizeMin=0.0006
Info    : Meshing 1D...
Info    : [  0%] Meshing curve 24 (Line)
Info    : [ 10%] Meshing curve 25 (Line)
Info    : [ 10%] Meshing curve 26 (Line)
Info    : [ 20%] Meshing curve 27 (Line)
Info    : [ 20%] Meshing curve 28 (Line)
Info    : [ 30%] Meshing curve 29 (Line)
Info    : [ 30%] Meshing curve 30 (Line)
Info    : [ 30%] Meshing curve 31 (Line)
Info    : [ 40%] Meshing curve 32 

Info    : Done meshing 2D (Wall 1.52485s, CPU 1.51258s)
Info    : Meshing 3D...
Info    : 3D Meshing 2 volumes with 1 connected component
Info    : Tetrahedrizing 8210 nodes...
Info    : Done tetrahedrizing 8218 nodes (Wall 0.0618073s, CPU 0.054968s)
Info    : Reconstructing mesh...
Info    :  - Creating surface mesh
Info    :  - Identifying boundary edges
Info    :  - Recovering boundary
Info    : Done reconstructing mesh (Wall 0.138549s, CPU 0.117196s)


Info    : Found volume 2
Info    : Found volume 1
Info    : It. 0 - 0 nodes created - worst tet radius 16.3058 (nodes removed 0 0)
Info    : It. 500 - 500 nodes created - worst tet radius 3.29323 (nodes removed 0 0)
Info    : It. 1000 - 1000 nodes created - worst tet radius 2.67174 (nodes removed 0 0)
Info    : It. 1500 - 1500 nodes created - worst tet radius 2.3705 (nodes removed 0 0)
Info    : It. 2000 - 2000 nodes created - worst tet radius 2.17552 (nodes removed 0 0)
Info    : It. 2500 - 2500 nodes created - worst tet radius 2.02593 (nodes removed 0 0)
Info    : It. 3000 - 3000 nodes created - worst tet radius 1.91331 (nodes removed 0 0)


Info    : It. 3500 - 3500 nodes created - worst tet radius 1.82822 (nodes removed 0 0)
Info    : It. 4000 - 4000 nodes created - worst tet radius 1.7568 (nodes removed 0 0)
Info    : It. 4500 - 4500 nodes created - worst tet radius 1.69512 (nodes removed 0 0)
Info    : It. 5000 - 5000 nodes created - worst tet radius 1.63961 (nodes removed 0 0)
Info    : It. 5500 - 5500 nodes created - worst tet radius 1.59198 (nodes removed 0 0)
Info    : It. 6000 - 6000 nodes created - worst tet radius 1.54684 (nodes removed 0 0)
Info    : It. 6500 - 6500 nodes created - worst tet radius 1.50836 (nodes removed 0 0)
Info    : It. 7000 - 7000 nodes created - worst tet radius 1.47404 (nodes removed 0 0)
Info    : It. 7500 - 7500 nodes created - worst tet radius 1.44266 (nodes removed 0 0)
Info    : It. 8000 - 8000 nodes created - worst tet radius 1.4139 (nodes removed 0 0)
Info    : It. 8500 - 8500 nodes created - worst tet radius 1.38781 (nodes removed 0 0)
Info    : It. 9000 - 9000 nodes created - wor

Info    : It. 11000 - 11000 nodes created - worst tet radius 1.27895 (nodes removed 0 0)
Info    : It. 11500 - 11500 nodes created - worst tet radius 1.26143 (nodes removed 0 0)
Info    : It. 12000 - 12000 nodes created - worst tet radius 1.24331 (nodes removed 0 0)
Info    : It. 12500 - 12500 nodes created - worst tet radius 1.22763 (nodes removed 0 0)
Info    : It. 13000 - 13000 nodes created - worst tet radius 1.21172 (nodes removed 0 0)
Info    : It. 13500 - 13500 nodes created - worst tet radius 1.19794 (nodes removed 0 0)
Info    : It. 14000 - 14000 nodes created - worst tet radius 1.18448 (nodes removed 0 0)
Info    : It. 14500 - 14500 nodes created - worst tet radius 1.17134 (nodes removed 0 0)
Info    : It. 15000 - 15000 nodes created - worst tet radius 1.15855 (nodes removed 0 0)
Info    : It. 15500 - 15500 nodes created - worst tet radius 1.14673 (nodes removed 0 0)
Info    : It. 16000 - 16000 nodes created - worst tet radius 1.13514 (nodes removed 0 0)
Info    : It. 16500 -

...

Info    : Total badness = 185098 
Info    : SwapImprove  


Info    : 311 swaps performed 
Info    : SwapImprove2  


Info    : 0 swaps performed 
Info    : ImproveMesh 
Info    : Total badness = 185005 


Info    : Total badness = 184797 
Info    : Done optimizing mesh (Wall 5.76905s, CPU 5.75609s)
Info    : Writing 'open_ended_antenna.msh'...


Mesh saved to open_ended_antenna.mshInfo    : Done writing 'open_ended_antenna.msh'

  Nodes: 30128
  Elements: 178626


### Mesh visualization.

In [6]:
# Render the physical groups. The air sphere is rendered transparent.
view_mesh(filename, transparent_groups= "air_sphere__None")

Loading mesh file: open_ended_antenna.msh
Groups to render transparent: air_sphere__None



Mesh loaded successfully with 2 cell blocks
Found 16412 triangles total
Physical group tags in mesh: {3: 'top_conductor', 4: 'ground_plane', 5: 'lumped_port', 6: 'air_sphere__None', 7: 'air_sphere__substrate'}


### Generate palace JSON config

In [7]:
config_filename: str = "op_spalace.conf"
freq_min: float = 3.0
freq_max: float = 3.5
freq_step: float = 0.005
eps_r: float = 2.2
loss_tan: float = 0.0009
port_impedance: float = 50.0
solver_order: int = 2

In [8]:
sim = Simulation()

def attr(name):
    return [pg_map[name]] if name in pg_map else []

sim.set_mesh_file(f"/work/{filename}")
sim.set_config_option("Problem.Output", "/work/results/open_ended_antenna/")

sim.set_config_option("Domains.Materials", [
    {
        "Attributes": attr("substrate"),
        "Permittivity": eps_r,
        "Permeability": 1.0,
        "LossTan": loss_tan
    },
    {
        "Attributes": attr("air"),
        "Permittivity": 1.0,
        "Permeability": 1.0
    }
])

sim.set_config_option("Boundaries.PEC", {
    "Attributes": attr("ground_plane") + attr("patch")
})

sim.set_config_option("Boundaries.LumpedPort", [
    {
        "Index": 1,
        "Attributes": attr("lumped_port"),
        "R": port_impedance,
        "Excitation": True,
        "Direction": "+Z"
    }
])
sim.set_config_option("Boundaries.Absorbing", {
    "Attributes": attr("farfield"),
    "Order": 1
})

sim.set_config_option("Solver.Order", solver_order)
sim.set_config_option("Solver.Driven.MinFreq", freq_min)
sim.set_config_option("Solver.Driven.MaxFreq", freq_max)
sim.set_config_option("Solver.Driven.FreqStep", freq_step)
sim.set_config_option("Solver.Driven.AdaptiveTol", 0.001)

config_path = sim.write_config(config_filename)
print(f"Wrote {config_path}")

Palace config written to op_spalace.conf
Wrote op_spalace.conf
